In [1]:
import json
import os
import sys
from collections import Counter

PROJECT_ROOT = os.getcwd()
if os.path.basename(PROJECT_ROOT) == "notebooks":
    PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from srcs.datasets.utils import load_valid_words, to_text
from srcs.datasets.vicocktail import load_vicocktail
from srcs.nlp.tokenizer import PhonemeTokenizer

d:\projects\VietnameseVSR\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
raw_dataset = load_vicocktail(
    split="all",
    seed=42,
    fraction=1.0,
    val_size=0.1,
    apply_filter=False,
)

for split, split_dataset in raw_dataset.items():
    print(f"{split}: {len(split_dataset):,} samples")

train: 174,665 samples
val: 19,408 samples
test: 1,167 samples


In [3]:
tokenizer = PhonemeTokenizer()
valid_words = load_valid_words()
word_frequencies = Counter()
lookup_candidates = {}

for split_dataset in raw_dataset.values():
    for label in split_dataset["label"]:
        for word in tokenizer.to_word(to_text(label)):
            if word not in valid_words:
                continue

            analysis = tokenizer.analyze(word)
            if not analysis["is_valid"]:
                continue

            initial = analysis["initial"]
            rhyme = tokenizer.merge_phoneme(
                [analysis["glide"], analysis["vowel"], analysis["final"]]
            )
            tone = analysis["tone"]
            key = tokenizer.merge_phoneme([initial, rhyme, tone])

            word_frequencies[word] += 1
            lookup_candidates.setdefault(key, set()).add(word)

lookup = {
    key: sorted(
        candidates,
        key=lambda word: (-word_frequencies[word], word),
    )
    for key, candidates in sorted(lookup_candidates.items())
}

In [ ]:
data_dir = os.path.join(PROJECT_ROOT, "srcs", "nlp", "data")
lookup_path = os.path.join(data_dir, "phoneme_lookup.json")

with open(lookup_path, "w", encoding="utf-8") as file:
    json.dump(lookup, file, ensure_ascii=False, indent=2)

ambiguous_groups = sum(len(candidates) > 1 for candidates in lookup.values())

print(f"Phoneme groups: {len(lookup):,}")
print(f"Ambiguous groups: {ambiguous_groups:,}")

Phoneme groups: 3,723
Ambiguous groups: 52
initial: 22
rhyme: 142
tone: 6


In [5]:
dataset = load_vicocktail(
    split="all",
    seed=42,
    fraction=1.0,
    val_size=0.1,
    min_word_frequency=5,
    apply_filter=True,
)

for split, split_dataset in dataset.items():
    removed = len(raw_dataset[split]) - len(split_dataset)
    
    print(
        f"{split}: {len(split_dataset):,} retained, "
        f"{removed:,} removed"
    )

train: 147,144 retained, 27,521 removed
val: 16,350 retained, 3,058 removed
test: 867 retained, 300 removed


In [6]:
train_components = {
    "initial": set(),
    "rhyme": set(),
    "tone": set(),
}

for label in dataset["train"]["label"]:
    for word in tokenizer.to_word(to_text(label)):
        analysis = tokenizer.analyze(word)

        if not analysis["is_valid"]:
            raise ValueError(f"Invalid training word: {word}")

        rhyme = tokenizer.merge_phoneme(
            [analysis["glide"], analysis["vowel"], analysis["final"]]
        )
        train_components["initial"].add(analysis["initial"])
        train_components["rhyme"].add(rhyme)
        train_components["tone"].add(analysis["tone"])

for name, tokens in train_components.items():
    path = os.path.join(data_dir, f"{name}.txt")

    with open(path, "w", encoding="utf-8") as file:
        file.write("\n".join(sorted(tokens)) + "\n")

for name, tokens in train_components.items():
    print(f"{name}: {len(tokens):,}")

from srcs.nlp.text_transform import PhonemeTransform

transform = PhonemeTransform()

print("Vocabulary sizes:", transform.vocab_size)

for label in dataset["train"]["label"][:5]:
    text = to_text(label)
    ids = transform.encode(text)
    
    print("Text:   ", text)
    print("IDs:    ", ids.tolist())
    print("Decoded:", transform.decode(ids))
    print()

Vocabulary sizes: {'initial': 22, 'rhyme': 142, 'tone': 6}
Text:    thì mình cũng
IDs:     [[16, 81, 1], [7, 77, 1], [5, 116, 3]]
Decoded: thì mình cũng

Text:    từ bậc ba con a hai con a một con a ba con bê hai con bê một con bê ba con xê hai con xê một con xê và một con đê
IDs:     [[15, 132, 1], [0, 41, 2], [0, 39, 4], [5, 92, 4], [11, 39, 4], [4, 31, 4], [5, 92, 4], [11, 39, 4], [7, 104, 2], [5, 92, 4], [11, 39, 4], [0, 39, 4], [5, 92, 4], [0, 71, 4], [4, 31, 4], [5, 92, 4], [0, 71, 4], [7, 104, 2], [5, 92, 4], [0, 71, 4], [0, 39, 4], [5, 92, 4], [14, 71, 4], [4, 31, 4], [5, 92, 4], [14, 71, 4], [7, 104, 2], [5, 92, 4], [14, 71, 4], [18, 39, 1], [7, 104, 2], [5, 92, 4], [2, 71, 4]]
Decoded: từ bậc ba con a hai con a một con a ba con bê hai con bê một con bê ba con xê hai con xê một con xê và một con đê

Text:    em thấy ở việt nam như này nè mưa người ta hay gọi là mưa rào việt nam nó ra của nó ngưng
IDs:     [[11, 57, 4], [16, 40, 5], [11, 111, 0], [18, 87, 2], [8, 33, 4], [10, 1

In [ ]:
from srcs.nlp.text_transform import PhonemeTransform

phoneme_transform = PhonemeTransform()
invalid_phonemes = Counter()
component_oov = {name: Counter() for name in phoneme_transform.component_names}
p2w_oov = Counter()
test_word_count = 0

for label in dataset["test"]["label"]:
    for word in tokenizer.to_word(to_text(label)):
        test_word_count += 1

        analysis = tokenizer.analyze(word)
        if not analysis["is_valid"]:
            invalid_phonemes[word] += 1
            continue

        phonemes = [
            analysis["initial"],
            tokenizer.merge_phoneme(
                [analysis["glide"], analysis["vowel"], analysis["final"]]
            ),
            analysis["tone"],
        ]

        for name, phoneme in zip(phoneme_transform.component_names, phonemes):
            if phoneme not in phoneme_transform.token2id[name]:
                component_oov[name][phoneme] += 1

        key = tokenizer.merge_phoneme(phonemes)
        if word not in lookup.get(key, []):
            p2w_oov[word] += 1

print(f"Test utterances: {len(dataset['test']):,}")
print(f"Test word occurrences: {test_word_count:,}")
print(
    f"Invalid phonemes: {len(invalid_phonemes):,} types / "
    f"{sum(invalid_phonemes.values()):,} occurrences"
)

for name, counter in component_oov.items():
    print(
        f"{name} OOV: {len(counter):,} types / "
        f"{sum(counter.values()):,} occurrences"
    )
    print(f"{name} OOV examples:", counter.most_common(20))

print(f"P2W OOV: {len(p2w_oov):,} types / {sum(p2w_oov.values()):,} occurrences")
print("P2W OOV examples:", p2w_oov.most_common(20))